# ML Lab Practical 4: Diabetes Prediction Using K-Nearest Neighbors (KNN)

## Aim
To classify whether a person has diabetes using the **K-Nearest Neighbors (KNN)** classification algorithm and evaluate the model using a confusion matrix, accuracy, error rate, precision, and recall.

**Dataset:** `diabetes.csv`  
**Target column:** `Outcome` (commonly 0 = non-diabetic and 1 = diabetic; verify the dataset's label definition).

## What this notebook covers
1. Understand the diabetes classification problem and the KNN algorithm.
2. Load and inspect the dataset.
3. Separate input features and the target.
4. Split the data into training and testing sets.
5. Standardize the features.
6. Train a KNN classifier with `k = 5`.
7. Predict the test-set outcomes.
8. Calculate and interpret the evaluation metrics.
9. Visualize the confusion matrix and answer viva questions.

**Important:** This notebook follows the workflow and main algorithm in the uploaded `DiabetesKNN.ipynb`. Optional diagnostic and visualization sections are added for learning. The original code does not perform missing-value imputation, zero-value correction, feature selection, or hyperparameter tuning; these are not silently added to the main model.

## 1. Theory: Diabetes Prediction as a Classification Problem

Diabetes is a medical condition in which blood glucose levels are higher than normal. A dataset for diabetes prediction typically contains patient measurements such as glucose level, blood pressure, body mass index (BMI), age, and other health-related attributes.

The task is a **binary classification problem** because the model predicts one of two classes:

- `0`: negative / no diabetes (subject to the dataset's label convention)
- `1`: positive / diabetes (subject to the dataset's label convention)

The model learns patterns from labelled training examples. It then predicts the class of records it has not seen during training.

### Features and target
- **Features (`X`):** the input columns used to make a prediction.
- **Target (`y`):** the `Outcome` column that contains the known class.

## 2. Theory: K-Nearest Neighbors (KNN)

KNN is a **supervised, instance-based classification algorithm**. It does not build a complicated equation during training. Instead, it stores the training examples and classifies a new example by finding the closest training examples.

### Working steps
1. Choose the value of `k`, the number of neighbours.
2. For a new test record, calculate its distance from the training records.
3. Select the `k` records with the smallest distances.
4. Count the class labels among those neighbours.
5. Assign the majority class to the new record.

### Example for k = 5
Suppose the five nearest neighbours have labels:

`[1, 1, 0, 1, 0]`

There are three neighbours from class 1 and two from class 0. KNN predicts class **1** by majority voting.

### Distance measure
For numeric features, KNN commonly uses Euclidean distance:

\[
d(P,Q)=\sqrt{\sum_{i=1}^{n}(P_i-Q_i)^2}
\]

Here, `P` and `Q` are two records and `n` is the number of features.

### What does k = 5 mean?
`n_neighbors=5` tells the model to consider the five closest training records when predicting a class.

- Very small `k` can make the model sensitive to noise.
- Very large `k` can smooth away useful local patterns.
- The best `k` depends on the dataset and should ideally be selected using validation or cross-validation.

The uploaded practical uses **k = 5**, so this notebook keeps that value in the main experiment.

## 3. Why Feature Scaling Is Important

KNN uses distances. If one feature has values in a much larger numerical range than another, it can dominate the distance calculation.

For example, a feature measured in thousands can outweigh a feature measured between 0 and 1. This can make the nearest-neighbour calculation misleading.

The practical uses `StandardScaler`, which standardizes each feature approximately to:
- mean = 0
- standard deviation = 1

The standardization formula is:

\[
z = \frac{x-\mu}{\sigma}
\]

where:
- `x` = original value
- `μ` = mean calculated from the training data
- `σ` = standard deviation calculated from the training data

**Important data-leakage rule:** fit the scaler only on the training data, then use that fitted scaler to transform the test data. The uploaded practical follows this rule.

## 4. Import Required Libraries

The following libraries are used:
- `pandas`: loading and handling tabular data.
- `train_test_split`: dividing data into training and testing subsets.
- `StandardScaler`: standardizing numeric features.
- `KNeighborsClassifier`: implementing the KNN algorithm.
- Metrics from `sklearn.metrics`: evaluating predictions.

In [ ]:
# 1. Import Libraries
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score
)

## 5. Load the Dataset

`pd.read_csv()` reads the CSV file into a pandas DataFrame.

The first five rows are displayed using `head()` so that we can inspect the data format and column names.

**Before running:** keep `diabetes.csv` in the same folder as this notebook, or replace the filename with the correct file path.

In [ ]:
# 2. Read the Dataset
data = pd.read_csv("diabetes.csv")

print("Dataset:")
print(data.head())

### Additional dataset inspection (recommended)

These checks help you understand the dataset before modelling:
- `shape`: number of rows and columns.
- `columns`: names of all columns.
- `info()`: data types and non-null counts.
- `isnull().sum()`: missing-value count per column.
- `describe()`: summary statistics for numeric columns.

These are inspection steps; they do not change the original practical's model workflow.

In [ ]:
# Inspect the dataset
print("Dataset shape:", data.shape)
print("\nColumn names:")
print(data.columns.tolist())

print("\nDataset information:")
data.info()

print("\nMissing values in each column:")
print(data.isnull().sum())

print("\nStatistical summary:")
display(data.describe())

### Dataset checks and important note about zero values

Confirm that:
- The target column is named exactly `Outcome`.
- The input columns are numeric, as expected by this practical.
- The target contains the expected class labels.

Some versions of the commonly used Pima Indians diabetes dataset use `0` in certain medical measurement columns to represent an unavailable or biologically implausible measurement. The uploaded practical does **not** replace those zeros or impute missing values. Therefore, this notebook preserves the original workflow. If your instructor expects zero-value treatment, it should be added as a separately documented preprocessing step rather than assumed.

In [ ]:
# Check target column and class distribution
if "Outcome" not in data.columns:
    raise ValueError("The dataset must contain a column named 'Outcome'.")

print("Outcome values:")
print(data["Outcome"].value_counts(dropna=False))

print("\nOutcome percentages:")
print((data["Outcome"].value_counts(normalize=True, dropna=False) * 100).round(2))

## 6. Select Features and Target

The uploaded code separates the dataset into:
- `X`: all columns except `Outcome`.
- `y`: only the `Outcome` column.

`drop("Outcome", axis=1)` removes the target column from the input features. `axis=1` means that a column is being removed.

The model must not receive the answer column as an input feature; otherwise, the evaluation would be invalid due to target leakage.

In [ ]:
# 3. Select Features and Target
X = data.drop("Outcome", axis=1)
y = data["Outcome"]

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nInput features:")
print(X.head())

## 7. Split the Dataset into Training and Testing Sets

The practical uses an 80:20 split:
- 80% of records for training.
- 20% of records for testing.

Parameters:
- `test_size=0.20`: reserves 20% for testing.
- `random_state=42`: makes the split reproducible.
- `stratify=y`: keeps approximately the same class proportions in both subsets.

The training set is used to fit the KNN model. The test set is held aside to evaluate how well the model predicts unseen records.

In [ ]:
# 4. Split the Dataset
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training feature shape:", X_train.shape)
print("Testing feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Testing target shape:", y_test.shape)

## 8. Normalize the Features Using StandardScaler

KNN calculates distances between records, so scaling is particularly important.

- `fit_transform(X_train)` learns the mean and standard deviation from the training features and scales them.
- `transform(X_test)` scales the test features using the training-set statistics.

Do not use `fit_transform()` on the test data: that would allow information from the test set to influence preprocessing.

In [ ]:
# 5. Normalize the Features
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Training data after scaling:")
print(X_train[:5])

## 9. Initialize and Train the KNN Classifier

`KNeighborsClassifier(n_neighbors=5)` creates a KNN classifier that uses the five nearest neighbours.

`knn.fit(X_train, y_train)` supplies the standardized training features and their correct labels to the model. For KNN, fitting primarily stores the training examples for later neighbour searches.

In [ ]:
# 6. Initialize and Train KNN
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

print("KNN model trained successfully.")

## 10. Predict the Test Data

`knn.predict(X_test)` predicts the class label for each record in the test set.

The predictions are stored in `y_pred`. They will be compared with the actual labels `y_test`.

In [ ]:
# 7. Predict the Test Data
y_pred = knn.predict(X_test)

print("First 10 actual labels:")
print(y_test.iloc[:10].to_list())

print("\nFirst 10 predicted labels:")
print(y_pred[:10])

## 11. Evaluation Metrics – Theory

### A. Confusion Matrix

A confusion matrix compares actual classes with predicted classes. For binary classification, it contains:

| Term | Meaning |
|---|---|
| True Negative (TN) | Actual 0, predicted 0 |
| False Positive (FP) | Actual 0, predicted 1 |
| False Negative (FN) | Actual 1, predicted 0 |
| True Positive (TP) | Actual 1, predicted 1 |

With scikit-learn's default label ordering `[0, 1]`, the matrix layout is:

\[
\begin{bmatrix}
TN & FP \\
FN & TP
\end{bmatrix}
\]

### B. Accuracy

Accuracy is the proportion of all predictions that are correct.

\[
Accuracy = \frac{TP+TN}{TP+TN+FP+FN}
\]

### C. Error Rate

Error rate is the proportion of predictions that are incorrect.

\[
Error\ Rate = 1 - Accuracy
\]

### D. Precision

Precision answers: **Of all records predicted as positive, how many were actually positive?**

\[
Precision = \frac{TP}{TP+FP}
\]

### E. Recall

Recall answers: **Of all actual positive records, how many did the model correctly identify?**

\[
Recall = \frac{TP}{TP+FN}
\]

For a diabetes-screening example, recall is important because a false negative means that a person who belongs to the positive class was predicted as negative. Accuracy alone may hide this type of error, especially when classes are imbalanced.

## 12. Calculate the Evaluation Metrics

This section follows the metric calculations in the uploaded practical:
- `confusion_matrix()` creates the confusion matrix.
- `accuracy_score()` calculates accuracy.
- `1 - accuracy` calculates error rate.
- `precision_score()` calculates precision.
- `recall_score()` calculates recall.
- `zero_division=0` avoids a division warning and returns 0 if a metric's denominator is zero.

For binary classification, scikit-learn treats label `1` as the positive class by default.

In [ ]:
# 8. Calculate Evaluation Metrics
cm = confusion_matrix(y_test, y_pred)
accuracy = accuracy_score(y_test, y_pred)
error_rate = 1 - accuracy
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

## 13. Display the Results

The following code prints the confusion matrix and the evaluation metrics. Values are rounded for readability.

**Your numerical results will depend on the actual `diabetes.csv` file.** No result is pre-filled or assumed in this notebook.

In [ ]:
# 9. Display Results
print("\nConfusion Matrix:")
print(cm)

print("\nAccuracy:", round(float(accuracy), 4))
print("Accuracy Percentage:", round(float(accuracy) * 100, 2), "%")
print("Error Rate:", round(float(error_rate), 4))
print("Error Rate Percentage:", round(float(error_rate) * 100, 2), "%")
print("Precision:", round(float(precision), 4))
print("Recall:", round(float(recall), 4))

## 14. Visualize the Confusion Matrix (Additional)

A heatmap makes the four confusion-matrix counts easier to interpret. The rows represent actual classes and the columns represent predicted classes.

This is an additional visualization; it does not change the trained model or the predictions.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Outcome 0", "Outcome 1"]
)
disp.plot(cmap="Blues", values_format="d")
plt.title("Confusion Matrix – KNN Diabetes Classification")
plt.show()

## 15. Optional: Classification Report and F1-Score

The uploaded practical reports accuracy, error rate, precision, and recall. The following optional section adds F1-score and a classification report for a more complete evaluation.

\[
F1 = 2 \times \frac{Precision \times Recall}{Precision + Recall}
\]

F1-score is the harmonic mean of precision and recall. It is useful when you want to balance these two measures. It is an additional metric, not a replacement for the metrics required by the original practical.

In [ ]:
from sklearn.metrics import f1_score, classification_report

f1 = f1_score(y_test, y_pred, zero_division=0)

print("F1-Score:", round(float(f1), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

## 16. Understanding the Result

After running the notebook, interpret your output as follows:

1. **Confusion matrix:** identify TN, FP, FN, and TP using the matrix layout.
2. **Accuracy:** tells you the overall fraction of correct predictions.
3. **Error rate:** tells you the fraction of incorrect predictions.
4. **Precision:** tells you how reliable positive predictions are.
5. **Recall:** tells you how many actual positive cases were detected.
6. **F1-score (optional):** combines precision and recall.

Do not claim that the model is suitable for medical diagnosis based only on this practical. It is an educational classification exercise, not a clinically validated diagnostic system.

## 17. Advantages and Limitations of KNN

### Advantages
- Simple and easy to understand.
- No complicated training procedure.
- Can be used for classification and regression variants.
- Can model non-linear decision boundaries.

### Limitations
- Prediction can be slow for large training datasets because distances must be calculated.
- Sensitive to feature scaling.
- Sensitive to noisy or irrelevant features.
- Performance depends on the choice of `k` and distance metric.
- Can be affected by imbalanced classes.

### Why StandardScaler is used here
Because KNN relies on distance, standardizing the features prevents variables with larger numeric ranges from dominating the distance calculation.

## 18. Common Errors and Troubleshooting

| Error / symptom | Likely reason | What to check |
|---|---|---|
| `FileNotFoundError` | `diabetes.csv` is not in the notebook's working folder | Put the CSV beside the notebook or provide its correct path |
| `KeyError: 'Outcome'` | Target column has a different name or spelling | Print `data.columns` and use the exact target column name |
| Conversion / input error | One or more feature columns contain text | Inspect `data.dtypes`; this practical expects numeric input features |
| Missing-value error | Dataset contains null values | Inspect `data.isnull().sum()` and follow the preprocessing method required by your instructor |
| Unexpected metric values | Different dataset version, preprocessing, or class labels | Verify the dataset, target labels, and exact split settings |
| Plotting library error | Matplotlib is not installed | Install it using `pip install matplotlib` |

### Run order
Run the notebook from top to bottom using **Kernel → Restart & Run All** (menu wording may vary). Make sure the CSV file is available before running.

## 19. Conclusion

In this practical, a K-Nearest Neighbors classifier is used to predict the `Outcome` class from the diabetes dataset. The data is divided into training and testing sets, the input features are standardized using `StandardScaler`, and a KNN model with five neighbours is trained. The model predicts the test records and is evaluated using a confusion matrix, accuracy, error rate, precision, and recall. An optional confusion-matrix plot, F1-score, and classification report provide additional insight into model performance.

**Fill in after execution:**  
The model achieved an accuracy of ______%, precision of ______, and recall of ______ on the test dataset.

## 20. Viva Questions and Answers

**Q1. What is the aim of this practical?**  
To predict the diabetes outcome using the K-Nearest Neighbors algorithm and evaluate the classifier.

**Q2. What is KNN?**  
KNN is a supervised machine-learning algorithm that predicts a class based on the majority class among the nearest training examples.

**Q3. Is KNN supervised or unsupervised?**  
Supervised, because it learns from labelled training data.

**Q4. Is this a classification or regression problem?**  
Classification, because the target `Outcome` represents discrete classes.

**Q5. What does K represent in KNN?**  
The number of nearest neighbours considered to make a prediction.

**Q6. What is the value of K in this practical?**  
5, as specified by `n_neighbors=5`.

**Q7. What is Euclidean distance?**  
A distance measure calculated as the square root of the sum of squared differences between corresponding feature values.

**Q8. Why is feature scaling required in KNN?**  
Because KNN uses distances, and features with larger scales can otherwise dominate the distance calculation.

**Q9. What is StandardScaler?**  
A preprocessing tool that standardizes features using the mean and standard deviation.

**Q10. Why do we fit the scaler only on training data?**  
To avoid data leakage from the test set.

**Q11. What is the purpose of `train_test_split()`?**  
It divides the dataset into training and testing subsets.

**Q12. What does `test_size=0.20` mean?**  
20% of the records are reserved for testing.

**Q13. What is the purpose of `random_state=42`?**  
It makes the random split reproducible.

**Q14. What is the purpose of `stratify=y`?**  
It preserves approximately the same target-class proportions in training and testing sets.

**Q15. What is `X`?**  
The input feature matrix, containing all columns except `Outcome`.

**Q16. What is `y`?**  
The target vector containing the `Outcome` labels.

**Q17. What does `fit()` do in KNN?**  
It stores the training examples and their labels for use during neighbour searches.

**Q18. What does `predict()` do?**  
It predicts class labels for the supplied input records.

**Q19. What is a confusion matrix?**  
A table comparing actual labels with predicted labels.

**Q20. Define True Positive.**  
An actual positive record correctly predicted as positive.

**Q21. Define True Negative.**  
An actual negative record correctly predicted as negative.

**Q22. Define False Positive.**  
An actual negative record incorrectly predicted as positive.

**Q23. Define False Negative.**  
An actual positive record incorrectly predicted as negative.

**Q24. What is accuracy?**  
The ratio of correct predictions to the total number of predictions.

**Q25. What is error rate?**  
The ratio of incorrect predictions to total predictions; it is equal to `1 - accuracy`.

**Q26. What is precision?**  
The proportion of predicted positive records that are actually positive.

**Q27. What is recall?**  
The proportion of actual positive records correctly identified by the model.

**Q28. Why can recall be important in diabetes screening?**  
Low recall means more actual positive cases may be missed (false negatives).

**Q29. What is the limitation of KNN on a large dataset?**  
Prediction may be slow because distances to many training examples must be calculated.

**Q30. What happens if K is too small?**  
The model can become sensitive to noise and overfit local examples.

**Q31. What happens if K is too large?**  
The model may become too smooth and underfit local patterns.

**Q32. Does this practical perform feature selection?**  
No. The uploaded code uses all columns except `Outcome` as input features.

**Q33. Does the uploaded practical handle missing values or zero-coded measurements?**  
No. It does not include imputation or zero-value correction.

**Q34. Is this model ready for clinical diagnosis?**  
No. It is an educational experiment and would require extensive clinical validation before any medical use.

## 21. Final Execution Checklist

- [ ] `diabetes.csv` is available in the correct folder.
- [ ] The CSV contains the `Outcome` column.
- [ ] Dataset inspection completed.
- [ ] Features and target selected correctly.
- [ ] Train-test split completed with 80:20 ratio.
- [ ] StandardScaler fitted on training data only.
- [ ] KNN model trained with `k = 5`.
- [ ] Test predictions generated.
- [ ] Confusion matrix and required metrics displayed.
- [ ] Optional visualization and F1-score executed.
- [ ] Actual output values copied into the conclusion.

**Practical 4 follows the uploaded notebook's core sequence and settings.** The additional inspection, visualization, F1-score, classification report, theory, troubleshooting, and viva sections are included to make the notebook easier to study and explain.